# Twitter Sentiment Classification (Fixed Version)This notebook keeps the same structure as the original one.Every change is marked with a `# NEW:` comment so it is easy to see what was added and why.**The three problems that were fixed:**1. The text cleaner deleted so many words that `"I will murder you"` became just `"murder"`.2. Rows with the same text but different labels were left in the data.3. The train/test split was random, but the dataset contains about 6 paraphrases of every   tweet under the same `ID`. Random splitting put paraphrases of the same tweet in both   train and test, so the test score was too optimistic.

In [ ]:
#### Requirment
# Pandas
# Numpy
# Matplotlib
# Seaborn
# Scikit-learn
# BM_NLP_Classification_WP
# Pickle
# streamlit

# 1. Import the libraries

In [ ]:
# Basic libraries
import re
import pandas as pd
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning tools from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

import BM_NLP_Classification_WP

import importlib
importlib.reload(BM_NLP_Classification_WP)

import pickle

# 2. Load the dataset

In [ ]:
# Read with header=None and give the four columns proper names
DF = pd.read_csv("twitter_training.csv",
                 header=None,
                 names=["ID", "Entity", "Sentiment", "Text"])

In [ ]:
DF.head()

# This is a multiclass NLP text-classification problem.

In [ ]:
# Number of rows and columns in the dataset
DF.shape

In [ ]:
# Column names, non-null counts and data types
DF.info()

In [ ]:
# Count the missing values in each column
DF.isna().sum()

In [ ]:
# Count how many rows are exact duplicates
DF.duplicated().sum()

In [ ]:
# Work on a copy so the original DataFrame stays untouched
df = DF.copy()

# 3. Explore the data (EDA)

In [ ]:
# To find out many tweets of each sentiment do we have
print(df["Sentiment"].value_counts())
print()
print("In percentage:")
print(df["Sentiment"].value_counts(normalize=True).round(3) * 100)

In [ ]:
# Bar plot of the sentiment
df["Sentiment"].value_counts().plot(kind="bar")

plt.title("How many tweets of each sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Number of tweets")
plt.xticks(rotation=45)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
# To find out how many different entities (topics) are there
print("Number of Entities:", df["Entity"].nunique())

# The 15 most common ones
top_entities = df["Entity"].value_counts().head(15)
sns.barplot(x=top_entities.values, y=top_entities.index)
plt.title("Top 15 entities by number of tweets")
plt.xlabel("Number of tweets")
plt.ylabel("Entity")
plt.show()

In [ ]:
# NEW: Check how many rows share the same ID.
# In this dataset one ID = one original tweet plus several paraphrases of it.
# This number is the reason we cannot use a normal random train/test split later.
rows_per_id = len(df) / df["ID"].nunique()

print("Number of rows      :", len(df))
print("Number of unique IDs:", df["ID"].nunique())
print("Rows per ID         :", round(rows_per_id, 2))

In [ ]:
# NEW: Look at one ID to see what those "paraphrase" rows actually look like
df[df["ID"] == 2401][["ID", "Entity", "Sentiment", "Text"]]

In [ ]:
# Look at a few example tweets for each sentiment
for label in ["Positive", "Negative", "Neutral", "Irrelevant"]:
    print("=" * 60)
    print(label)
    print("=" * 60)
    examples = df[df["Sentiment"] == label]["Text"].dropna().head(3)
    for tweet in examples:
        print(tweet)

# 4. Clean the data

In [ ]:
# Remove duplicated rows
df = df.drop_duplicates()

# Remove rows where the tweet text is missing
df = df.dropna(subset=["Text"])

# Reset the row numbers after deleting rows
df = df.reset_index(drop=True)

print("Rows and columns after cleaning:", df.shape)

# 4.1 Fix rows that have the same text but different labelsIn the original notebook we only looked at these rows and then deleted one single ID.That did not fix anything, because there are about 2000 such rows.Now we do two things:1. Give every identical text the **same** label (the label that appears most often for it).2. Keep only **one copy** of each identical text.Step 2 matters: if the exact same sentence appears in both the training set and the testset, the model can pass the test by remembering instead of by learning.

In [ ]:
# Make a "comparison version" of the text: lowercase, single spaces, no space at the ends.
# Two tweets that differ only by capitalisation or spacing will now look identical.
df["Text_Check"] = (
    df["Text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
# Count how many different labels each text has
label_counts = df.groupby("Text_Check")["Sentiment"].nunique()

# The texts that have more than one label are the problem ones
conflicting_texts = label_counts[label_counts > 1].index

print("Texts with more than one label:", len(conflicting_texts))
print("Rows involved in the conflict :", df["Text_Check"].isin(conflicting_texts).sum())

In [ ]:
# Show a few of the conflicting rows so we can see the problem with our own eyes
conflicts = df[df["Text_Check"].isin(conflicting_texts)].sort_values("Text_Check")

conflicts[["ID", "Entity", "Sentiment", "Text"]].head(20)

In [ ]:
# NEW: Give every identical text one single label - the most common label for that text.
# .mode() returns the most frequent value(s), and [0] takes the first one.
majority_label = (
    df.groupby("Text_Check")["Sentiment"]
      .agg(lambda labels: labels.mode()[0])
)

df["Sentiment"] = df["Text_Check"].map(majority_label)

print("Texts with more than one label now:",
      (df.groupby("Text_Check")["Sentiment"].nunique() > 1).sum())

In [ ]:
# NEW: Keep only one row per unique text.
# This removes the exact-copy rows that would otherwise land in both train and test.
rows_before = len(df)

df = df.drop_duplicates(subset="Text_Check").reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after :", len(df))
print("Removed    :", rows_before - len(df))

# 4.2 Clean the text itselfThis is where the biggest bug was.The old stopword list contained words like `i`, `you`, `will`, `me`, `my`, and the old codealso threw away every word shorter than 3 letters. So a sentence like:```"I will murder you"   ->   "murder"```The model was given **one single word** and had no chance to understand the sentence.The new version keeps those small but meaningful words.

In [ ]:
# NEW: A much shorter stopword list.
# We only remove words that really carry no meaning (articles, prepositions, "to be", ...).
# We deliberately KEEP: i, me, you, we, will, can, not, no, but, very, too ...
# because in a short tweet those words change the meaning a lot.
stopword_text = """
a an the and or of at by for with about into to from in on off over under then there
is am are was were be been being have has had having do does did doing
this that these those
"""

stopwords = set(stopword_text.split())
print("Number of stopwords:", len(stopwords))

In [ ]:
# Make a function to clean the Text
def clean_text(text):
    """Take one tweet and return a cleaned version of it."""

    # 1. Make everything lowercase
    text = text.lower()

    # 2. Remove web links
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # 3. Remove @mentions (like @netflix)
    text = re.sub(r"@\w+", " ", text)

    # 4. NEW: Remove apostrophes, so "don't" becomes "dont" instead of "don" + "t"
    text = re.sub(r"'", "", text)

    # 5. NEW: Turn ! and ? into words before we delete the punctuation.
    #    "This is terrible!!!" then keeps the information that the person shouted.
    text = re.sub(r"!+", " exclaim ", text)
    text = re.sub(r"\?+", " question ", text)

    # 6. Keep only letters and spaces (removes numbers, emojis, punctuation)
    text = re.sub(r"[^a-z\s]", " ", text)

    # 7. NEW: Drop stopwords, but only drop single letters (before: shorter than 3 letters).
    #    This keeps real sentiment words like bad, sad, wow, fun, win, lol, omg.
    words = text.split()
    words = [w for w in words if w not in stopwords and len(w) > 1]

    # 8. Join the words back into one sentence
    return " ".join(words)

In [ ]:
# Quick test on one sentence
sample = "I LOVE @Borderlands 3!!! Best game ever, check http://bl3.com :)"
print("Before:", sample)
print("After :", clean_text(sample))

In [ ]:
# NEW: Check the sentences that used to break the model.
# Before the fix these became a single word. Now the sentence survives.
for sentence in ["I will murder you",
                 "I will kill you all",
                 "I hate this game",
                 "This game is not good",
                 "I love this game"]:
    print(f"{sentence:25} -> {clean_text(sentence)}")

In [ ]:
# Apply the cleaning function to every tweet
df["Clean_Text"] = df["Text"].apply(clean_text)

# Show the original and the cleaned version side by side
df[["Text", "Clean_Text", "Sentiment"]].head(10)

In [ ]:
# Find out empty tweets (for example a tweet with only a link)
empty_rows = (df["Clean_Text"].str.strip() == "").sum()
print("Empty tweets after cleaning:", empty_rows)

In [ ]:
# Removing these rows because there is nothing left to learn
df = df[df["Clean_Text"].str.strip() != ""].reset_index(drop=True)
print("Final number of rows:", df.shape[0])

In [ ]:
# Find out how long are the tweets?
def count_words(text):
    return len(text.split())

df["word_count"] = df["Clean_Text"].apply(count_words)

In [ ]:
# Basic statistics of the tweet length (mean, min, max, quartiles)
df["word_count"].describe()

In [ ]:
# Draw a histplot of word length of the tweets
plt.hist(df["word_count"], bins=50)
plt.title("Number of words per cleaned tweet")
plt.xlabel("Words")
plt.ylabel("Number of tweets")
plt.show()

In [ ]:
# The most frequent words in the whole dataset
all_words = " ".join(df["Clean_Text"]).split()
word_freq = pd.Series(all_words).value_counts().head(20)

In [ ]:
# Draw a barplot of the most frequent words in the whole dataset
sns.barplot(x=word_freq.values, y=word_freq.index)
plt.title("Top 20 most frequent words")
plt.xlabel("Count")
plt.ylabel("Word")
plt.show()

# 5. Split the data into training and testing sets**This is the second important fix.**`train_test_split` picks random *rows*. But this dataset stores one tweet as about 6 rows(the original plus paraphrases), all sharing the same `ID`. A random split would put"im getting on borderlands and i will murder you all" in the training set and"im coming on borderlands and i will murder you all" in the test set.The model would then get a high score for recognising a sentence it had almost alreadyseen, and the score would not tell us how it behaves on a real new tweet.So we split by **ID** instead of by row: all rows of one ID go together, either all intotraining or all into testing.

In [ ]:
# NEW: Split by ID (a "group split") instead of by row.

# 1. Take the list of all unique IDs
unique_ids = df["ID"].unique()

# 2. Shuffle them (random_state keeps the result the same on every run)
rng = np.random.RandomState(42)
rng.shuffle(unique_ids)

# 3. Put 20% of the IDs into the test set
n_test_ids = int(len(unique_ids) * 0.2)
test_ids = set(unique_ids[:n_test_ids])

# 4. A row goes to test if its ID is a test ID
is_test = df["ID"].isin(test_ids)

# 5. NEW: Shuffle the training rows.
#    Cross-validation cuts the data into blocks in the order we give it.
#    Our rows are still sorted by ID and Entity, so without this shuffle every
#    block would hold a different set of games and the CV score would be far
#    too pessimistic. (The old notebook got this for free, because
#    train_test_split shuffles the rows itself.)
#    (.copy() is needed because the row numbers come back read-only,
#     and shuffle has to be able to rearrange them.)
train_rows = df.index[~is_test].to_numpy().copy()
rng.shuffle(train_rows)

test_rows = df.index[is_test].to_numpy().copy()

X_train = df.loc[train_rows, "Clean_Text"]
X_test  = df.loc[test_rows,  "Clean_Text"]

Y_train = df.loc[train_rows, "Sentiment"]
Y_test  = df.loc[test_rows,  "Sentiment"]

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

In [ ]:
# NEW: Safety check - no ID is allowed to appear in both sets.
# If this prints 0, there is no leakage between training and testing.
train_ids = set(df.loc[~is_test, "ID"])
print("IDs in both train and test:", len(train_ids & test_ids))

In [ ]:
# NEW: Check that both sets still have a similar mix of the four sentiments.
# (A group split cannot be stratified, so it is worth looking at.)
print("Train:")
print(Y_train.value_counts(normalize=True).round(3))
print()
print("Test:")
print(Y_test.value_counts(normalize=True).round(3))

# 6. Text to number

In [ ]:
# TF-IDF turns each tweet into a numeric vector
# max_features: keep only the 20000 most useful terms
# ngram_range=(1, 2): use single words and word pairs
# min_df=3: ignore terms that appear in fewer than 3 tweets
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=3,

    # NEW: sublinear_tf softens the effect of a word being repeated many times
    # inside the same tweet, which helps with shouty tweets.
    sublinear_tf=True
)

# Convert the vocabulary of train data
X_train_t = tfidf.fit_transform(X_train)

# Convert the vocabulary of test data
X_test_t = tfidf.transform(X_test)

In [ ]:
# print the shape of train and test data
print("Training matrix shape:", X_train_t.shape)
print("Testing matrix shape :", X_test_t.shape)

# 7. Train the modelsThe test score here will most likely be **lower** than the 0.85 of the old notebook.That is expected, and it is a good sign: the old number was helped by the paraphrasesleaking into the test set. This new number is the honest one.

In [ ]:
# NEW: Build the 5 cross-validation folds ourselves, using the same idea as section 5.
#
# This is the same problem as section 5, one level deeper.
# If we just say cv=5, the folds are cut without looking at the IDs, so the
# paraphrases of one tweet end up in different folds. Cross-validation then
# grades the model on sentences it has almost already seen and reports about
# 0.89 when the real score is about 0.55.
#
# The rule is the same as before: all rows of one ID must stay together.

# The ID of every training row (same order as X_train / Y_train)
train_ids_array = df.loc[train_rows, "ID"].to_numpy()

# Take the training IDs and shuffle them
cv_ids = np.unique(train_ids_array)
rng.shuffle(cv_ids)

# Deal those IDs into 5 piles of roughly equal size
id_piles = np.array_split(cv_ids, 5)

# For each pile: that pile becomes the validation part of one fold,
# and all the other rows become the training part of that fold.
# scikit-learn wants row positions (0, 1, 2, ...), not the DataFrame row labels.
row_positions = np.arange(len(train_ids_array))

group_folds = []

for pile in id_piles:
    is_val = np.isin(train_ids_array, pile)
    group_folds.append((row_positions[~is_val], row_positions[is_val]))

print("Number of folds:", len(group_folds))
print("Validation rows per fold:", [len(val) for train, val in group_folds])

In [ ]:
# NEW: Safety check - the same ID must never be in the training part and the
# validation part of the same fold. All five numbers below should be 0.
for fold_number, (train_part, val_part) in enumerate(group_folds):
    shared = set(train_ids_array[train_part]) & set(train_ids_array[val_part])
    print("Fold", fold_number, "- IDs on both sides:", len(shared))

In [ ]:
# Train and compare several classifiers with cross-validation,
# then tune the best one and evaluate it on the test set
result = BM_NLP_Classification_WP.universal_nlp_classifier(
    X_train_t,
    Y_train,
    X_test=X_test_t,
    y_test=Y_test,

    # NEW: use our own ID-based folds instead of a plain cv=5
    cv=group_folds
)

In [ ]:
# Name of the model that scored best
result["best_model_name"]

In [ ]:
# Hyper-parameters found by the tuning step
result["best_params"]

In [ ]:
# Cross-validation scores of every model that was tried
result["results"]

In [ ]:
# Keep the fitted best model in a variable
best_model = result["model"]

In [ ]:
# Predict on the training data to check how well the model fits it
train_pred = best_model.predict(X_train_t)

print(classification_report(Y_train, train_pred))

In [ ]:
# Compare train and test F1 to check for overfitting
print("Train F1:", f1_score(Y_train, train_pred, average="weighted"))
print("Test F1 :", f1_score(Y_test, result["test_pred"], average="weighted"))

In [ ]:
# Plot the confusion matrix of the test predictions
ConfusionMatrixDisplay.from_predictions(Y_test, result["test_pred"])

plt.title("Test Confusion Matrix")
plt.show()

# 8. Check the predictions on new sentencesThis is the check that failed in the old notebook.

In [ ]:
# NEW: The same test sentences as before, but now we also print the cleaned text,
# so we can always see exactly what the model was given.
tests = [
    "I will murder you",
    "I will kill you",
    "I hate this game",
    "This game is terrible",
    "This is the worst game ever",
    "I love this game",
    "This game is amazing",
    "This game is not good",
    "This game is not bad at all"
]

for text in tests:
    cleaned = clean_text(text)
    vector = tfidf.transform([cleaned])
    prediction = best_model.predict(vector)[0]

    print(f"{text:30} | {cleaned:28} -> {prediction}")

### A word of warning about "murder" and "kill"Some of these may still not come out as `Negative`, and that is not a bug in the code.In this dataset those words are mostly used as gaming slang, for example*"xbox really killing it"* or *"that boss killed me lol"*. The model can only learnwhat the data shows it, and the next cell shows exactly what the data shows it.

In [ ]:
# NEW: Look at how the training data actually labels these words.
for word in ["murder", "kill", "hate", "love"]:
    rows = df[df["Text"].str.contains(rf"\b{word}\b", case=False, na=False, regex=True)]

    print("\nWORD:", word, " (found in", len(rows), "tweets)")
    print(rows["Sentiment"].value_counts())

# 9. Save the modelThe files are saved with a `_v2` name, so the old files (and the Streamlit app that usesthem) keep working until you decide to switch over.When you switch, remember to also copy the **new stopword list and the new `clean_text`function** into `app.py`. The app has to clean the text in exactly the same way as thisnotebook, otherwise the predictions will be wrong.

In [ ]:
# NEW: saved under a new name so the old model file is not overwritten
with open("sentiment_nlp_model_v2.pkl", "wb") as file:
    pickle.dump(best_model, file)

In [ ]:
# Save the fitted vectorizer too, it is needed to transform new tweets
with open("tfidf_vectorizer_v2.pkl", "wb") as file:
    pickle.dump(tfidf, file)

In [ ]:
# NEW: Reload the saved files and predict once, to be sure the saving worked
with open("sentiment_nlp_model_v2.pkl", "rb") as file:
    check_model = pickle.load(file)

with open("tfidf_vectorizer_v2.pkl", "rb") as file:
    check_tfidf = pickle.load(file)

sentence = "I love this game so much"
print(sentence, "->",
      check_model.predict(check_tfidf.transform([clean_text(sentence)]))[0])